# circuit — the quantum grid, on Qiskit

This is where the enhanced algorithm actually runs. Everything the classical side produces — the
one-step Kraus operators $M_k(\Delta t)$ for a Markovian map, the fixed companion operator $E$ for a
non-Markovian one — is turned into **one** unitary, compiled into **one** Qiskit circuit, and that same
circuit is executed over and over on the Aer simulator. The last measured state is fed back in as the
next input. No operator is ever rebuilt.

**Sz.-Nagy dilation** (thesis `dilate`, `figure6.17.py`; cf. Head-Marsden *et al.*, Phys. Rev. Research
**3**, 013182 (2021)): a linear operator $A$ with $\|A\|\le1$ is not unitary, so it cannot be a circuit
by itself. It becomes the top-left block of a unitary one qubit larger,
$$U=\begin{pmatrix}A & \sqrt{I-AA^\dagger}\\ \sqrt{I-A^\dagger A} & -A^\dagger\end{pmatrix},$$
so preparing $|0\rangle_{\rm anc}\!\otimes\!|x\rangle$, applying $U$ and post-selecting the ancilla on
$|0\rangle$ leaves $A|x\rangle$ (up to the norm, which is restored classically — the `normfac`
bookkeeping of the thesis). The ancilla is the highest-index qubit, so the ancilla-$|0\rangle$ branch is
exactly the first half of the statevector.

**Registers used here:**
- **Markovian (Lindblad):** the state on the grid is $\mathrm{vec}(\rho)$, 16 amplitudes = 4 qubits, +1
  ancilla = 5 qubits. Each Kraus operator $M_k$ enters as the superoperator $M_k^*\!\otimes\!M_k$
  (because $\mathrm{vec}(M\rho M^\dagger)=(M^*\!\otimes\!M)\,\mathrm{vec}(\rho)$); the operator sum is
  the sum over the circuits, exactly as in `figure6.17.py`.
- **Non-Markovian (path integral, HEOM):** the state is the memory register $X=(\mathrm{vec}\rho_n,
  \dots,\mathrm{vec}\rho_{n-K+1})$, $16K$ amplitudes padded to the next power of two, +1 ancilla. For
  $K=25$ that is $400\to512$ = 9 qubits, +1 = 10 qubits.

**Note on gate synthesis:** the dilated operator is applied as a single `UnitaryGate`. Decomposing an
arbitrary $n$-qubit unitary into 1- and 2-qubit gates costs $O(4^n)$ and is a separate
(hardware-compilation) problem; it is not what this algorithm is about, so the gate is handed to the
simulator as a unitary instruction.

In [ ]:
import numpy as np
from scipy.linalg import sqrtm
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

import import_ipynb                      # lets `import` read .ipynb modules
from grid import (vec, unvec, enhanced_kraus, transfer_tensors,
                  companion_propagator, load_maps)

_SIM = AerSimulator(method="statevector")


def _pow2(dim):
    return 1 << int(np.ceil(np.log2(dim)))

## The dilated channel
$$\text{vec}(\rho_S(dt)) = \text{vec}\left(\sum_k M_k(dt)\rho_S(0)M_k^\dagger(dt)\right) = \sum_k \left(M_k^*(dt) \otimes M_k(dt)\right)\text{vec}(\rho_S(0)).$$

$$ = U_{M_k^* \otimes M_k} (\vert{}\text{vec}(\rho_S(0))\rangle \otimes \vert{}0\rangle) \big |_0 = \begin{pmatrix} M_k^* \otimes M_k & D_{M_k^* \otimes M_k}^\dagger \\ D_{M_k^* \otimes M_k} & -(M_k^* \otimes M_k)^\dagger \end{pmatrix} \cdot \begin{pmatrix} \rho_1 \\ \vdots \\ \rho_N \\ 0 \\ \vdots \\ 0 \end{pmatrix} \Bigg |_0 = \begin{pmatrix} M_k^* \otimes M_k \vert{}\text{vec}(\rho_S(0))\rangle \\ D_{{M_k^* \otimes M_k}} \vert{}\text{vec}(\rho_S(0))\rangle \end{pmatrix} \Big |_0 = \text{vec}(\rho_S(dt)) $$

Wobei $\big |_0$ bedeutet dass wir nur die obere Hälfte des Vektors nehmen. Um Schreibaufwand zu sparen haben wir bei $M_k(dt)$ das Argument $dt$ weggelassen.

In [ ]:
class DilatedChannel:
    """One linear operator -> one unitary -> one reusable Qiskit circuit.

    Built once; `apply` executes the circuit and returns the operator acting
    on the input, together with the ancilla post-selection probability (the
    quantity that decides how many shots a real device would need)."""

    def __init__(self, A, label="A"):
        A = np.asarray(A, complex)      # objekt das der Transfer tensor zurück gibt
        self.dim = A.shape[0]
        self.pad = _pow2(self.dim)      # wenn dim nicht 2^n ist, wird auf die nächstgrößere 2^n gesetzt. dim=9 -> pad=16
        self.n_qubits = int(np.log2(self.pad)) + 1    #  Zahl der System-Qubits ist log_2(pad). + 1 Hilfsqubit (Ancilla) für Sz.-Nagy-Dilation 

        Ap = np.zeros((self.pad, self.pad), complex)    # leeres array für normiertes A. Ap ist (2^n,2^n) damait es auf circuit passt
        Ap[:self.dim, :self.dim] = A                    # A wird links oben in leeres array gegeben
        self.s = float(np.linalg.norm(Ap, 2)) * (1 + 1e-12)   # calc ||A|| since A must be normalized for Sz.-Nagy-Dilation
        As = Ap / self.s                                # Normalize Ap by division through s

        I = np.eye(self.pad)                            # Build Sz.-Nagy Dilation of As
        B = sqrtm(I - As @ As.conj().T)
        C = sqrtm(I - As.conj().T @ As)
        B, C = 0.5 * (B + B.conj().T), 0.5 * (C + C.conj().T)
        self.U = np.block([[As, B], [C, -As.conj().T]])
        self.gate = UnitaryGate(self.U, label=label)      # save U as unitary gate for qiskit circuit
        self.n_circuits = 0                               # no circuits executed yet

    def circuit(self, x):
        """The circuit executed at every step: load the current state, apply
        the ONE gate, read out. Only the loaded state changes."""
        v = np.zeros(2 * self.pad, complex)     # Vektor v mit doppelter Dim von pad (wegen Ancilla-Qubits) wird erstellt.
        v[:self.dim] = np.asarray(x, complex) / np.linalg.norm(x)   # Aktueller Zustand x wird auf 1 normiert und in obere Hälfte von v geschrieben. |v> = |0> \otimes |x>
        qc = QuantumCircuit(self.n_qubits, name="grid step")        # qc wird erstellt mit n_qubits = log_2(pad) + 1 (Ancilla-Qubit)
        qc.set_statevector(Statevector(v))                          # Zustand v wird als Startzustand in qc geladen. 
        qc.append(self.gate, range(self.n_qubits))                  # U wird auf qc gesetzt. U ist die Sz.-Nagy-Dilation von A, die in __init__ erstellt wurde.
        qc.save_statevector()                                       # Der Zustand nach Anwendung von U soll gespeichert werden, damit er später ausgelesen werden kann
        return qc

    def apply(self, x, backend=None):
        """Run one circuit. Returns (A @ x, ancilla-|0> success probability)."""
        nrm = np.linalg.norm(x)     # norm of input state, to restore later

        # self.circuit(x) baut den Qiskit-Schaltkreis für diesen Schritt und lädt den Zustand x
        # (backend or _SIM) wählt das Backend. Wenn kein backend übergeben wurde, wird der standardmäßige AerSimulator (_SIM) genutzt.
        # .run(...).result().get_statevector() startet die Simulation und gibt den finalen, vollen Zustandsvektor zurück
        sv = np.asarray((backend or _SIM).run(self.circuit(x)).result().get_statevector())
    
        self.n_circuits += 1    # zählt wie viele circuits erstellt wurden
        top = sv[:self.pad]     # ancilla is |0> -> extract the top half of the statevector which is the part corresponding to the system qubits
        p_success = float(np.vdot(top, top).real)   # Wslkeit, mit der Hilfsqubit bei echter Messung in |0> landen würde. Nur dann erhalten wir den Zustand des Systems.

        # Weil A am Anfang eventuell mit Nullen auf eine Zweierpotenz aufgefüllt wurde, schneidet top[:self.dim] den eventuell künstlich erweiterten Teil wieder ab. 
        # Danach wird mit (self.s * nrm) multipliziert. Damit wird sowohl die Division durch den Skalierungsfaktor $s$ (aus dem Konstruktor) als auch die Division 
        # durch die ursprüngliche Norm nrm klassisch wieder rückgängig gemacht.
        statevec = top[:self.dim] * (self.s * nrm)
        return statevec, p_success   # restore the norm

    def unitarity_error(self):  # Berechnet wie weit U von Sz.-Nagy-Dilation von einer unitären Matrix entfernt ist
        return float(np.abs(self.U.conj().T @ self.U
                            - np.eye(self.U.shape[0])).max())

## Markovian map on the grid: reuse the one-step Kraus operators $M_k(\Delta t)$

In [ ]:
def populations_kraus_qiskit(L_dt, d, rho0, n_steps, backend=None):
    """Enhanced algorithm, Markovian case, on real circuits.

    M_k(dt) is built once from L(dt); each becomes one dilated circuit, and
    every step is the operator sum over those circuits with the output fed
    back in. Returns dict: pops, rho, channels, n_kraus, n_qubits,
    n_circuits, p_success (min over the run)."""

    # Liouville-Propagator L_dt wird in seine Kraus-Operatoren M_k zerlegt. min_eig ist kleinste EW der Choi-Matrix, der Positivität des Kanals überprüft.
    kraus, min_eig = enhanced_kraus(L_dt, d)        

    # Baue ein DilatedChannel-Objekt für jeden Kraus-Operator. Jeder Kraus-Operator wird einmal in eine unitäre Matrix U dilatiert, die dann auf qc gesetzt wird.
    chans = [DilatedChannel(np.kron(M.conj(), M), f"M{k}")     # Operator np.kron(M.conj(), M) wird auf qc gesetzt
             for k, M in enumerate(kraus)]

    # Berechnet die relative "Stärke" (Gewichtung) jedes einzelnen Kraus-Zweiges. Dies dient später zur Identifikation dominanter 
    # physikalischer Kanäle im Gegensatz zu numerisch vernachlässigbaren Rausch-Kanälen.
    weight = np.array([np.linalg.norm(M, 2) ** 2 for M in kraus])
    weight /= weight.max()

    v = vec(np.asarray(rho0, complex))      # Start-Dichtematrix rho0 wird vektorisiert. v ist Input für ersten Schritt. v = |0> \otimes |rho0>
    traj = [np.asarray(rho0, complex)]      # Liste aller Dichtematrizen über die Zeit, startwert ist rho0
    p_last =np.ones(len(chans))             # p_last: Array aus 1, das für jeden Kraus-Kanal die letzte gemessene Ancilla-Erfolgswahrscheinlichkeit abspeichert
    
    for _ in range(n_steps):
        out = np.zeros_like(v)              # Für jeden neuen Zeitschritt wird ein Vektor out vorbereitet, in dem die Ergebnisse der einzelnen Schaltkreise aufsummiert werden.
        for k, ch in enumerate(chans):      # operator sum = sum of circuits, berechnet: sum_k M_k rho M_k^\dagger
            y, p_last[k] = ch.apply(v, backend)    # y ist der statevector nach Anwendung des Kraus-Kanals k auf den Input v.
            out += y                        # addiert den transformierten Zustandsvektor des Kanals zum Gesamtzustand auf
        v = out                             # out wird zum neuen Zustand v für den nächsten Zeitschritt
        traj.append(unvec(v, d))            # Vektor wird in quadratische d*d Dichtematrix zurück-transformiert und an Trajektorie angehängt.

    traj = np.array(traj)                   # Die Liste der Dichtematrizen wird in ein NumPy-Array umgewandelt.

    # Es wird eine Maske major definiert, die alle physikalisch relevanten Kanäle herausfiltert (Gewicht > 10^-8), um die statistische 
    # Erfolgswahrscheinlichkeit nicht durch unbedeutende Rausch-Kanäle zu verfälschen.
    major = weight > 1e-8

    return dict(pops=np.real(np.diagonal(traj, axis1=1, axis2=2)), rho=traj, # pops: populations, rho: full density matrices for each time step
                channels=chans, n_kraus=len(kraus), kraus_dt=kraus, # channels: list of DilatedChannel objects, n_kraus: number of Kraus operators, kraus_dt: list of Kraus operators
                min_choi_eig=min_eig, n_qubits=chans[0].n_qubits,   # n_qubits: Anzahl der genutzten Qubits pro Schaltkreis.
                n_circuits=sum(c.n_circuits for c in chans),        # n_circuits: Die Gesamtzahl aller simulierten Schaltkreis-Ausführungen.
                kraus_weight=weight, p_per_branch=p_last,           # kraus_weight: relative weight of each Kraus channel, p_per_branch: last ancilla success probability for each Kraus channel
                p_success=float(p_last[major].min()), n_major=int(major.sum()), # p_success: Die minimale Erfolgswahrscheinlichkeit unter den physikalisch relevanten Hauptkanälen (wichtig zur Abschätzung des Shot-Bedarfs auf echter Hardware).
                unitarity=max(c.unitarity_error() for c in chans))  # unitarity: Maximale numerische Unitaritätsfehler über alle Kanäle hinweg.

## Non-Markovian map on the grid: reuse the companion operator $E$

In [ ]:
def populations_memory_qiskit(maps_in, rho0, K, n_steps, backend=None):
    """Enhanced algorithm, non-Markovian case, on real circuits.

    The transfer tensors T_1..T_K and the companion operator E are built once
    from the short-time maps; E becomes ONE dilated circuit that is executed
    once per step, with the memory register read out and fed back in.
    Returns dict: pops, rho, channel, E, T_norms, K, n_qubits, n_circuits,
    p_success (min over the run)."""

    # Falls maps_in ein Dateipfad (String) ist, wird über eine Hilfsfunktion (load_maps) die entsprechende Datei mit den zuvor berechneten 
    # Abbildungen von der Festplatte geladen.
    if isinstance(maps_in, str):
        maps_in = load_maps(maps_in)

    # Falls die geladenen Daten als Dictionary strukturiert sind, wird das Array unter dem Schlüssel "maps" extrahiert. 
    # Andernfalls wird die Eingabe direkt in ein standardmäßiges NumPy-Array konvertiert. Maps enthält K+1 objekte der dimension (d², d²)
    maps = maps_in["maps"] if isinstance(maps_in, dict) else np.asarray(maps_in)

    # D ist die Dimension des vektorisierten Raums (vec(rho) hat die Dimension d^2). Daraus wird durch die Wurzel np.sqrt(D) 
    # die physikalische Dimension d des Quantensystems berechnet.
    D = maps.shape[1]
    d = int(round(np.sqrt(D)))

    # für die Berechnung der Transfertensoren bis zur Gedächtnistiefe K müssen genau so viele Zeitschritte an Eingangsabbildungen vorliegen
    if K > len(maps) - 1:
        raise ValueError(f"K={K} needs maps up to index K (have {len(maps)-1})")

    # Aus den dynamischen Abbildungen werden die Transfertensoren T_1,...,T_K berechnet. Diese beschreiben die Gedächtniseffekte
    T = transfer_tensors(maps, K)

    # Baut aus den Transfertensoren den Companion-Operator E auf. Dieser Operator schiebt das Gedächtnisregister in jedem Schritt 
    # um eine Position weiter und berechnet gleichzeitig den neuesten Zustand.
    E = companion_propagator(T)                 # the ONE operator

    # nimmt diesen sehr großen Operator und bettet ihn über die Sz.-Nagy-Dilation in ein einzigen wiederverwendbaren Schaltkreis (chan) ein
    chan = DilatedChannel(E, "E")               # the ONE circuit

    v0 = vec(np.asarray(rho0, complex))                         # Der Startzustand rho0 wird vektorisiert (v0).
    hist = [maps[n] @ v0 for n in range(min(K, n_steps + 1))]   # Die ersten K Zeitschritte werden berechnet (oder n_steps+1 falls n_steps < K). hist speichert die vektorisierten Zustände für die ersten K Zeitschritte
    states = list(hist)                                         # states wird als Liste initialisiert, um die gesamte Trajektorie aufzubewahren

    # Die Vektoren der vergangenen Zustände werden in umgekehrter Reihenfolge (hist[::-1]) zu einem langen Vektor aneinandergereiht 
    # Das ist der Vektor, der als Zustand in den Quantenschaltkreis geladen wird und auf den der Companion-Operator E angewendet wird.
    X = np.concatenate(hist[::-1])
    p_min = 1.0     # p_min wird mit 1.0 initialisiert, umminimal gemessene Ancilla-Erfolgswslkeit über den gesamten Lauf aufzuzeichnen.

    # Diese Schleife simuliert die Dynamik ab dem Zeitpunkt, ab dem das Gedächtnisregister voll besetzt ist
    for _ in range(K, n_steps + 1):
        X, p = chan.apply(X, backend)   # Berechnet X_n = E @ X_{n-1} und die Erfolgswslkeit p, dass das Hilfsqubit in |0> gemessen wird.
        p_min = min(p_min, p)           # hält fest, wie niedrig die Erfolgswahrscheinlichkeit im schlechtesten Schritt absinkt.
        states.append(X[:D].copy())     # schneidet allerneuesten Zustand (ersten D Einträge) heraus und hängt ihn an Zusatndsliste an

    # Alle Zustandsvektoren aus states werden mit unvec wieder in d*d Dichtematrizen zurückverwandelt und als Trajektorie (traj) zusammengefasst
    traj = np.array([unvec(s, d) for s in states[:n_steps + 1]])

    return dict(pops=np.real(np.diagonal(traj, axis1=1, axis2=2)), rho=traj,
                channel=chan, E=E, K=K,
                T_norms=np.array([np.linalg.norm(Tm, 2) for Tm in T]),  # T_norms: Die Normen der Transfertensoren (zeigt, wie schnell das Gedächtnis des Systems mit der Zeit abklingt).
                n_qubits=chan.n_qubits, n_circuits=chan.n_circuits,     # n_qubits / n_circuits: Die Anzahl der benötigten Qubits (System + Gedächtnis + 1 Ancilla) sowie die Anzahl der Schaltkreis-Aufrufe.
                p_success=p_min, unitarity=chan.unitarity_error())      # p_success: Die minimale Erfolgswahrscheinlichkeit über die gesamte Simulation.